# Week 4 – Text Embeddings

Generating and Comparing Text Embeddings using Sentence Transformers.

## Objective
Generate embeddings for 20 sentences, calculate cosine similarity, identify the most similar pairs, and understand the role of embeddings in semantic search and RAG.

In [1]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../dataset/generated_text.csv")
df.head()

,text
0,Artificial intelligence helps computers perfor...
1,Machine learning allows systems to learn patte...
2,Natural language processing enables computers ...
3,Large language models can generate text by pre...
4,Text embeddings represent sentences as numeric...


In [3]:
text_col = next(
    (c for c in ["Generated_Text", "generated_text", "text", "Text", "sentence", "Sentence"] if c in df.columns),
    None
)
if text_col is None:
    raise ValueError("No text column found.")
sentences = df[text_col].dropna().astype(str).str.strip()
sentences = sentences[sentences.str.len() > 0].drop_duplicates().tolist()[:20]
print("Number of sentences:", len(sentences))
assert len(sentences) >= 20, "At least 20 sentences are required."


Number of sentences: 20


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences, convert_to_numpy=True, normalize_embeddings=True)
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (20, 384)
Embedding dimension: 384


In [5]:
np.save("../dataset/embeddings.npy", embeddings)
print("Saved embeddings.npy")

Saved embeddings.npy


In [6]:
similarity_matrix = cosine_similarity(embeddings)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"S{i+1}" for i in range(len(sentences))],
    columns=[f"S{i+1}" for i in range(len(sentences))]
)
similarity_df.round(3)

,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,S11,S12,S13,S14,S15,S16,S17,S18,S19,S20
S1,1.000,0.513,0.550,0.199,0.278,0.238,0.237,0.136,0.342,0.256,0.174,0.111,0.147,0.437,0.302,0.565,0.281,0.184,0.261,0.155
S2,0.513,1.000,0.414,0.376,0.206,0.221,0.243,0.177,0.417,0.213,0.255,0.104,0.109,0.205,0.484,0.472,0.379,0.269,0.276,0.139
S3,0.550,0.414,1.000,0.447,0.464,0.423,0.390,0.326,0.463,0.336,0.225,0.107,0.240,0.254,0.188,0.293,0.321,0.354,0.445,0.253
S4,0.199,0.376,0.447,1.000,0.491,0.426,0.368,0.508,0.396,0.371,0.301,0.110,0.321,0.190,0.262,0.132,0.298,0.386,0.490,0.329
S5,0.278,0.206,0.464,0.491,1.000,0.474,0.355,0.366,0.253,0.457,0.463,0.257,0.392,0.259,0.106,0.166,0.247,0.542,0.311,0.578
S6,0.238,0.221,0.423,0.426,0.474,1.000,0.399,0.377,0.312,0.394,0.232,0.258,0.499,0.317,0.100,0.121,0.179,0.359,0.641,0.326
S7,0.237,0.243,0.390,0.368,0.355,0.399,1.000,0.395,0.264,0.290,0.426,0.207,0.402,0.243,0.095,0.183,0.314,0.237,0.316,0.479
S8,0.136,0.177,0.326,0.508,0.366,0.377,0.395,1.000,0.209,0.326,0.388,0.104,0.276,0.091,0.139,0.030,0.288,0.288,0.361,0.513
S9,0.342,0.417,0.463,0.396,0.253,0.312,0.264,0.209,1.000,0.545,0.182,0.001,0.113,0.181,0.341,0.246,0.176,0.200,0.223,0.221
S10,0.256,0.213,0.336,0.371,0.457,0.394,0.290,0.326,0.545,1.000,0.390,0.239,0.265,0.214,0.156,0.141,0.278,0.469,0.249,0.429


In [7]:
pairs = []
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        pairs.append({
            "sentence_1_id": i+1,
            "sentence_2_id": j+1,
            "sentence_1": sentences[i],
            "sentence_2": sentences[j],
            "cosine_similarity": similarity_matrix[i, j]
        })
scores = pd.DataFrame(pairs).sort_values("cosine_similarity", ascending=False)
scores.to_csv("../dataset/similarity_scores.csv", index=False)
top5 = scores.head(5)
top5

,sentence_1_id,sentence_2_id,sentence_1,sentence_2,cosine_similarity
154,12,13,Cosine similarity measures how closely two vec...,A high cosine similarity score usually indicat...,0.659800
97,6,19,Sentence transformers are useful for comparing...,Transformers use attention mechanisms to captu...,0.641272
84,5,20,Text embeddings represent sentences as numeric...,"Embeddings are important for semantic search, ...",0.578025
14,1,16,Artificial intelligence helps computers perfor...,Cybersecurity systems can use artificial intel...,0.565419
1,1,3,Artificial intelligence helps computers perfor...,Natural language processing enables computers ...,0.550083


In [8]:
top5.to_csv("../dataset/sentence_pairs.csv", index=False)
print("Saved similarity_scores.csv and sentence_pairs.csv")

Saved similarity_scores.csv and sentence_pairs.csv


## Observations
- Similar sentences should generally have higher cosine similarity.
- Unrelated sentences should generally have lower similarity.
- The embedding dimension is shown by `embeddings.shape[1]`.
- Embeddings are useful for semantic search, document retrieval, recommendation systems, and RAG.